In [2]:
import urllib.request

url = ("https://api.statbank.dk/v1/data/FOLK1A/CSV"
       "?delimiter=Semicolon"
       "&OMR%C3%85DE=*"
       "&K%C3%98N=TOT"
       "&ALDER=IALT"
       "&Tid=2026K1")

urllib.request.urlretrieve(url, "folk1a_population.csv")
print("Hentet!")

Hentet!


In [3]:
import pandas as pd

# Læs den hentede fil
pop = pd.read_csv("folk1a_population.csv", sep=";", encoding="utf-8")
print(pop.head())
print(pop.columns.tolist())

               OMRÅDE    KØN        ALDER     TID  INDHOLD
0         Hele landet  I alt  Alder i alt  2026K1  6025603
1  Region Hovedstaden  I alt  Alder i alt  2026K1  1947212
2           København  I alt  Alder i alt  2026K1   671714
3       Frederiksberg  I alt  Alder i alt  2026K1   106150
4              Dragør  I alt  Alder i alt  2026K1    14474
['OMRÅDE', 'KØN', 'ALDER', 'TID', 'INDHOLD']


In [4]:
# Fjern "Hele landet" og "Region ..."-rækker — behold kun kommuner
pop_muni = pop[~pop['OMRÅDE'].str.startswith(('Hele landet', 'Region'))].copy()
pop_muni = pop_muni.rename(columns={'OMRÅDE': 'kommune_name', 'INDHOLD': 'population'})
pop_muni = pop_muni[['kommune_name', 'population']]

# Indlæs den eksisterende fil med kommunekoder
muni = pd.read_csv("dk_municipalities.csv")

# Merge på navn — opdater befolkningstal
muni = muni.drop(columns=['population_approx'])
muni = muni.merge(pop_muni, on='kommune_name', how='left')

# Tjek
print(f"Matchede kommuner: {muni['population'].notna().sum()} / {len(muni)}")
missing = muni[muni['population'].isna()]
if len(missing) > 0:
    print(f"Mangler match for: {missing['kommune_name'].tolist()}")

# Gem
muni.to_csv("dk_municipalities.csv", index=False)
print(f"\n✓ dk_municipalities.csv opdateret med officielle 2026K1-tal")
print(f"  Samlet befolkning: {muni['population'].sum():,.0f}")
muni.head(10)

Matchede kommuner: 98 / 98

✓ dk_municipalities.csv opdateret med officielle 2026K1-tal
  Samlet befolkning: 6,025,511


,kommune_kode,kommune_name,region,population
0,101,København,Hovedstaden,671714
1,147,Frederiksberg,Hovedstaden,106150
2,151,Ballerup,Hovedstaden,53950
3,153,Brøndby,Hovedstaden,40990
4,155,Dragør,Hovedstaden,14474
5,157,Gentofte,Hovedstaden,75106
6,159,Gladsaxe,Hovedstaden,71185
7,161,Glostrup,Hovedstaden,25834
8,163,Herlev,Hovedstaden,32030
9,165,Albertslund,Hovedstaden,29262
